In [2]:
import pandas as pd

# Show all columns when displaying records
pd.set_option("display.max_columns", None)

# Load the dataset
df = pd.read_csv("LMS_Dirty_Dataset.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [ ]:
#display first 10 records
print("First 10 records:")
display(df.head(10))
#display last 10 records
print("Last 10 records:")
display(df.tail(10))
#display the dataset shape
print("Dataset shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

#display column names 
print("Column names:")

for column in df.columns:
    print(column)


#display data types
print("Data types:")
print(df.dtypes)


#display sumary 
print("Summary statistics of numerical columns:")
display(df.describe())



In [ ]:
#so that i do not have to download again and again. it will just create the copy of original dataset. so that my dataset will be safe.
cleaned_df = df.copy()
#identify dublicates
duplicate_count = cleaned_df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

#remove duplicates
cleaned_df = cleaned_df.drop_duplicates()

# Reset the row index(important step after removing duplicates we must do it, i learned it from youtube.)
cleaned_df = cleaned_df.reset_index(drop=True)

print("Shape after removing duplicates:", cleaned_df.shape)

#detect missing values
print("Missing values in each column:")
print(cleaned_df.isnull().sum())

#handle missing values
attendance_median = cleaned_df["attendance_percent"].median()

cleaned_df["attendance_percent"] = (
    cleaned_df["attendance_percent"]
    .fillna(attendance_median)
)

print(
    "Missing attendance values replaced with median:",
    attendance_median
)

#virify
print("Missing values after treatment:")
print(cleaned_df.isnull().sum())

print(
    "Total missing values:",
    cleaned_df.isnull().sum().sum()
)


#jconvert data types
cleaned_df["enrollment_date"] = pd.to_datetime(
    cleaned_df["enrollment_date"],
    errors="coerce"
)

numeric_columns = [
    "student_id",
    "age",
    "course_fee",
    "discount",
    "attendance_percent",
    "assignment_score",
    "final_score",
    "rating"
]

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

cleaned_df["phone"] = cleaned_df["phone"].astype("string")
print("Data types after conversion:") # to print and check the data types after conversion
print(cleaned_df.dtypes)

#remove unnecessary xt based columns
text_columns = [
    "student_name",
    "course",
    "city",
    "gender",
    "payment_status",
    "mentor",
    "phone",
    "email",
    "completion_status"
]

for column in text_columns:
    cleaned_df[column] = (
        cleaned_df[column]
        .astype("string")
        .str.strip()
    )


# standarize capatilization
title_case_columns = [
    "student_name",
    "city",
    "gender",
    "payment_status",
    "mentor",
    "completion_status"
]

for column in title_case_columns:
    cleaned_df[column] = (
        cleaned_df[column]
        .str.title()
    )



#standardize city names
cleaned_df["city"] = cleaned_df["city"].replace({
    "ktm": "Kathmandu",
    "Ktm": "Kathmandu",
    "KTM": "Kathmandu"
})

print("Unique city names:")
print(cleaned_df["city"].unique())

#standardize course names
cleaned_df["city"] = cleaned_df["city"].replace({
    "ktm": "Kathmandu",
    "Ktm": "Kathmandu",
    "KTM": "Kathmandu"
})

print("Unique city names:")
print(cleaned_df["city"].unique())

#standardize email
cleaned_df["email"] = cleaned_df["email"].str.lower()


#detect negative invalid values
required_columns = [
    "course_fee",
    "attendance_percent",
    "assignment_score",
    "final_score"
]

negative_records = cleaned_df[
    (cleaned_df[required_columns] < 0).any(axis=1)
]

print("Records containing invalid negative values:")
display(negative_records)


#correct negative values
for column in required_columns:

    negative_mask = cleaned_df[column] < 0

    negative_count = negative_mask.sum()

    if negative_count > 0:

        valid_median = cleaned_df.loc[
            cleaned_df[column] >= 0,
            column
        ].median()

        cleaned_df.loc[
            negative_mask,
            column
        ] = valid_median

        print(
            column,
            "- Replaced",
            negative_count,
            "negative value(s) with median:",
            valid_median
        )




#print the final cleaned dataset

print(
    "Remaining duplicates:",
    cleaned_df.duplicated().sum()
)

print(
    "Remaining missing values:",
    cleaned_df.isnull().sum().sum()
)

print(
    "Remaining negative values:",
    (cleaned_df[required_columns] < 0).sum().sum()
)

display(cleaned_df.head())    







In [ ]:
#outlier detection using IQR method

required_columns = [
    "course_fee",
    "attendance_percent",
    "assignment_score",
    "final_score"
]

# Store IQR calculations
iqr_results = []

# Store outlier conditions for later use
iqr_outlier_masks = {}

print("=" * 80)
print("IQR OUTLIER DETECTION")
print("=" * 80)

for column in required_columns:

    # Calculate Q1 and Q3
    Q1 = cleaned_df[column].quantile(0.25)
    Q3 = cleaned_df[column].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Calculate lower and upper bounds
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    # Detect outliers
    outlier_mask = (
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    )

    outlier_records = cleaned_df[outlier_mask]

    # Save the outlier condition
    iqr_outlier_masks[column] = outlier_mask

    # Save the calculated information
    iqr_results.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outlier_records)
    })

    # Display calculation for each column
    print("\n" + "-" * 80)
    print("Column:", column)
    print("-" * 80)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Number of outliers:", len(outlier_records))

    print("\nDetected outlier records:")

    if outlier_records.empty:
        print("No outliers detected.")
    else:
        display(outlier_records)

# Convert the IQR calculations to a DataFrame
iqr_summary = pd.DataFrame(iqr_results)

print("\n" + "=" * 80)
print("IQR CALCULATION SUMMARY")
print("=" * 80)
display(iqr_summary)

# Combine the outlier conditions
combined_iqr_mask = pd.Series(
    False,
    index=cleaned_df.index
)

for column in required_columns:
    combined_iqr_mask = (
        combined_iqr_mask |
        iqr_outlier_masks[column]
    )

# Display all unique records that are outliers
all_iqr_outliers = cleaned_df[combined_iqr_mask]

print("\n" + "=" * 80)
print("ALL UNIQUE IQR OUTLIER RECORDS")
print("=" * 80)

if all_iqr_outliers.empty:
    print("No IQR outlier records were detected.")
else:
    display(all_iqr_outliers)

print(
    "Total unique IQR outlier records:",
    len(all_iqr_outliers)
)

In [ ]:
# outlier handling
required_columns = [
    "course_fee",
    "attendance_percent",
    "assignment_score",
    "final_score"
]

# Recalculate and store the IQR bounds
iqr_bounds = {}
combined_outlier_mask = pd.Series(
    False,
    index=cleaned_df.index
)

for column in required_columns:

    Q1 = cleaned_df[column].quantile(0.25)
    Q3 = cleaned_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    iqr_bounds[column] = {
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    }

    column_outlier_mask = (
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    )

    combined_outlier_mask = (
        combined_outlier_mask |
        column_outlier_mask
    )


print("=" * 80)
print("METHOD 1: REMOVE OUTLIER RECORDS")
print("=" * 80)

# Create a separate copy and remove all IQR outlier records
df_outliers_removed = cleaned_df[
    ~combined_outlier_mask
].copy()

df_outliers_removed = (
    df_outliers_removed
    .reset_index(drop=True)
)

print("Original cleaned dataset shape:", cleaned_df.shape)
print(
    "Shape after removing outliers:",
    df_outliers_removed.shape
)
print(
    "Number of records removed:",
    len(cleaned_df) - len(df_outliers_removed)
)

display(df_outliers_removed.head())


print("\n" + "=" * 80)
print("METHOD 2: REPLACE OUTLIERS WITH COLUMN MEDIAN")
print("=" * 80)

# Use a separate copy for median replacement
df_median_replaced = cleaned_df.copy()

for column in required_columns:

    lower_bound = iqr_bounds[column]["Lower Bound"]
    upper_bound = iqr_bounds[column]["Upper Bound"]

    outlier_mask = (
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    )

    column_median = cleaned_df[column].median()
    outlier_count = outlier_mask.sum()

    df_median_replaced.loc[
        outlier_mask,
        column
    ] = column_median

    print(
        f"{column}: {outlier_count} outlier(s) replaced "
        f"with median {column_median}"
    )

print(
    "Shape after median replacement:",
    df_median_replaced.shape
)

display(df_median_replaced.head())


print("\n" + "=" * 80)
print("METHOD 3: WINSORIZATION USING clip()")
print("=" * 80)

# Use a separate copy for Winsorization
df_winsorized = cleaned_df.copy()

for column in required_columns:

    lower_bound = iqr_bounds[column]["Lower Bound"]
    upper_bound = iqr_bounds[column]["Upper Bound"]

    outlier_count = (
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    ).sum()

    # Cap values at the lower and upper IQR boundaries
    df_winsorized[column] = (
        df_winsorized[column]
        .clip(
            lower=lower_bound,
            upper=upper_bound
        )
    )

    print(
        f"{column}: {outlier_count} value(s) Winsorized "
        f"between {lower_bound} and {upper_bound}"
    )

print(
    "Shape after Winsorization:",
    df_winsorized.shape
)

display(df_winsorized.head())


print("\n" + "=" * 80)
print("OUTLIER-HANDLING COMPARISON")
print("=" * 80)

handling_comparison = pd.DataFrame({
    "Technique": [
        "Remove outlier records",
        "Replace outliers with median",
        "Winsorization using clip()"
    ],
    "Rows before handling": [
        len(cleaned_df),
        len(cleaned_df),
        len(cleaned_df)
    ],
    "Rows after handling": [
        len(df_outliers_removed),
        len(df_median_replaced),
        len(df_winsorized)
    ],
    "Rows removed": [
        len(cleaned_df) - len(df_outliers_removed),
        0,
        0
    ]
})

display(handling_comparison)

In [7]:
#outlier detection using Z-score method

required_columns = [
    "course_fee",
    "attendance_percent",
    "assignment_score",
    "final_score"
]

# Use the cleaned data before applying IQR outlier handling
df_zscore = cleaned_df.copy()

# Store the names of the created Z-score columns
calculated_zscore_columns = []

print("=" * 80)
print("Z-SCORE CALCULATION")
print("=" * 80)

for column in required_columns:

    # Calculate the mean
    column_mean = df_zscore[column].mean()

    # Calculate population standard deviation
    column_standard_deviation = (
        df_zscore[column].std(ddof=0)
    )

    # Create the Z-score column
    zscore_column = column + "_zscore"

    df_zscore[zscore_column] = (
        df_zscore[column] - column_mean
    ) / column_standard_deviation

    calculated_zscore_columns.append(zscore_column)

    print("\nColumn:", column)
    print("Mean:", column_mean)
    print(
        "Standard Deviation:",
        column_standard_deviation
    )

print("\nZ-scores calculated successfully.")


print("\n" + "=" * 80)
print("FIRST 10 RECORDS WITH Z-SCORES")
print("=" * 80)

zscore_display_columns = [
    "student_id",
    "student_name",
    "course_fee",
    "course_fee_zscore",
    "attendance_percent",
    "attendance_percent_zscore",
    "assignment_score",
    "assignment_score_zscore",
    "final_score",
    "final_score_zscore"
]

display(df_zscore[zscore_display_columns].head(10))


print("\n" + "=" * 80)
print("Z-SCORE OUTLIERS FOR EACH COLUMN")
print("=" * 80)

zscore_summary = []

for column in required_columns:

    zscore_column = column + "_zscore"

    # Detect records where Z-score is greater than 3 or less than -3
    column_zscore_mask = (
        (df_zscore[zscore_column] > 3) |
        (df_zscore[zscore_column] < -3)
    )

    column_zscore_outliers = df_zscore[
        column_zscore_mask
    ]

    print("\n" + "-" * 80)
    print("Column:", column)
    print("-" * 80)
    print(
        "Number of Z-score outliers:",
        len(column_zscore_outliers)
    )

    if column_zscore_outliers.empty:
        print("No Z-score outliers detected.")
    else:
        display(
            column_zscore_outliers[
                [
                    "student_id",
                    "student_name",
                    column,
                    zscore_column
                ]
            ]
        )

    # Save summary information
    zscore_summary.append({
        "Column": column,
        "Mean": df_zscore[column].mean(),
        "Standard Deviation": (
            df_zscore[column].std(ddof=0)
        ),
        "Minimum Z-Score": (
            df_zscore[zscore_column].min()
        ),
        "Maximum Z-Score": (
            df_zscore[zscore_column].max()
        ),
        "Outlier Count": len(column_zscore_outliers)
    })


# Combine all Z-score outlier conditions
combined_zscore_mask = pd.Series(
    False,
    index=df_zscore.index
)

for zscore_column in calculated_zscore_columns:

    combined_zscore_mask = (
        combined_zscore_mask |
        (df_zscore[zscore_column] > 3) |
        (df_zscore[zscore_column] < -3)
    )

# Display complete records where any Z-score is outside -3 to 3
all_zscore_outliers = df_zscore[
    combined_zscore_mask
]

print("\n" + "=" * 80)
print("ALL RECORDS WHERE Z-SCORE IS GREATER THAN 3 OR LESS THAN -3")
print("=" * 80)

if all_zscore_outliers.empty:
    print("No Z-score outlier records were detected.")
else:
    display(all_zscore_outliers)

print(
    "Total Z-score outlier records:",
    len(all_zscore_outliers)
)


# Display Z-score summary
zscore_summary_df = pd.DataFrame(zscore_summary)

print("\n" + "=" * 80)
print("Z-SCORE SUMMARY")
print("=" * 80)
display(zscore_summary_df)

Z-SCORE CALCULATION

Column: course_fee
Mean: 10119.997142857143
Standard Deviation: 53027.404781220626

Column: attendance_percent
Mean: 70.15428571428572
Standard Deviation: 16.650153887871305

Column: assignment_score
Mean: 67.22285714285714
Standard Deviation: 20.294588803271612

Column: final_score
Mean: 65.01142857142857
Standard Deviation: 20.527330512250824

Z-scores calculated successfully.

FIRST 10 RECORDS WITH Z-SCORES


,student_id,student_name,course_fee,course_fee_zscore,attendance_percent,attendance_percent_zscore,assignment_score,assignment_score_zscore,final_score,final_score_zscore
0,1,Gita,10000,-0.002263,73.0,0.170912,46,-1.045740,98,1.607056
1,2,Nabin,10000,-0.002263,68.0,-0.129385,31,-1.784853,63,-0.097988
2,3,Gita,5000,-0.096554,71.0,0.050793,31,-1.784853,99,1.655772
3,4,Gita,7000,-0.058837,88.0,1.071805,43,-1.193562,38,-1.315876
4,5,Gita,5000,-0.096554,100.0,1.792519,89,1.073052,98,1.607056
5,6,Anisha,7000,-0.058837,57.0,-0.790040,70,0.136842,70,0.243021
6,7,Suman,5000,-0.096554,52.0,-1.090337,38,-1.439933,39,-1.267161
7,8,Sita,5000,-0.096554,71.0,0.050793,38,-1.439933,98,1.607056
8,9,Gita,7000,-0.058837,70.0,-0.009266,88,1.023777,98,1.607056
9,10,Sita,5000,-0.096554,53.0,-1.030278,88,1.023777,47,-0.877436



Z-SCORE OUTLIERS FOR EACH COLUMN

--------------------------------------------------------------------------------
Column: course_fee
--------------------------------------------------------------------------------
Number of Z-score outliers: 1


,student_id,student_name,course_fee,course_fee_zscore
18,19,Sita,999999,18.66731



--------------------------------------------------------------------------------
Column: attendance_percent
--------------------------------------------------------------------------------
Number of Z-score outliers: 0
No Z-score outliers detected.

--------------------------------------------------------------------------------
Column: assignment_score
--------------------------------------------------------------------------------
Number of Z-score outliers: 0
No Z-score outliers detected.

--------------------------------------------------------------------------------
Column: final_score
--------------------------------------------------------------------------------
Number of Z-score outliers: 0
No Z-score outliers detected.

ALL RECORDS WHERE Z-SCORE IS GREATER THAN 3 OR LESS THAN -3


,student_id,student_name,course,city,age,gender,enrollment_date,course_fee,discount,payment_status,attendance_percent,assignment_score,final_score,rating,mentor,phone,email,completion_status,course_fee_zscore,attendance_percent_zscore,assignment_score_zscore,final_score_zscore
18,19,Sita,Flutter,Kathmandu,24,Male,2026-05-09,999999,10,Pending,41.0,75,51,4,Ishan,9803681891,user18@gmail.com,Ongoing,18.66731,-1.750992,0.383213,-0.682574


Total Z-score outlier records: 1

Z-SCORE SUMMARY


,Column,Mean,Standard Deviation,Minimum Z-Score,Maximum Z-Score,Outlier Count
0,course_fee,10119.997143,53027.404781,-0.096554,18.667310,1
1,attendance_percent,70.154286,16.650154,-1.811051,1.792519,0
2,assignment_score,67.222857,20.294589,-1.834127,1.615068,0
3,final_score,65.011429,20.527331,-1.705601,1.704487,0
